In [4]:
import gc
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

def process_data(expose_size: float = 1/2):
    user_log = pd.read_csv('data/2/user_log_format1.csv')
    user_info = pd.read_csv('data/2/user_info_format1.csv')
    train = pd.read_csv('data/2/train_format1.csv')
    test = pd.read_csv('data/2/test_format1.csv')

    user_log.rename(columns={"seller_id": "merchant_id"}, inplace=True)

    user_info["age_range"].fillna(0, inplace=True)
    user_info["gender"].fillna(2, inplace=True)
    user_log["brand_id"].fillna(0, inplace=True)

    train_main, train_ratio = train_test_split(
        train,
        test_size=expose_size,
        random_state=42,
        stratify=train['label']
    )

    del train
    gc.collect()

    train_main["origin"] = "train"
    test["origin"] = "test"
    data = pd.concat([train_main, test], sort=False)
    data = data.drop(["prob"], axis=1)

    del train_main, test
    gc.collect()

    merchant_label_ratio = train_ratio.groupby('merchant_id')['label'].mean().reset_index().rename(
        columns={'label': 'merchant_label_ratio'})
    data = data.merge(merchant_label_ratio, on='merchant_id', how='left')

    item_label_ratio_max = pd.merge(user_log[(user_log['action_type'] == 2)], train_ratio,
                                    how='inner').drop_duplicates()
    item_label_ratio_mean = item_label_ratio_max.groupby(['item_id'])['label'].mean().reset_index().rename(
        columns={'label': 'item_label_ratio'})
    item_label_ratio_max = pd.merge(item_label_ratio_max, item_label_ratio_mean, how='left')
    del item_label_ratio_mean
    gc.collect()
    item_label_ratio_max = item_label_ratio_max.groupby(['merchant_id'])['item_label_ratio'].max().reset_index()

    cat_label_ratio_max = pd.merge(user_log[(user_log['action_type'] == 2)], train_ratio, how='inner').drop_duplicates()
    cat_label_ratio_mean = cat_label_ratio_max.groupby(['cat_id'])['label'].mean().reset_index().rename(
        columns={'label': 'cat_label_ratio'})
    cat_label_ratio_max = pd.merge(cat_label_ratio_max, cat_label_ratio_mean, how='inner')
    del cat_label_ratio_mean
    gc.collect()
    cat_label_ratio_max = cat_label_ratio_max.groupby(['merchant_id'])['cat_label_ratio'].max().reset_index()

    brand_label_ratio_max = pd.merge(user_log[(user_log['action_type'] == 2)], train_ratio,
                                     how='inner').drop_duplicates().dropna()
    brand_label_ratio_mean = brand_label_ratio_max.groupby(['brand_id'])['label'].mean().reset_index().rename(
        columns={'label': 'brand_label_ratio'})
    brand_label_ratio_max = pd.merge(brand_label_ratio_max, brand_label_ratio_mean, how='left')
    del brand_label_ratio_mean
    gc.collect()
    brand_label_ratio_max = brand_label_ratio_max.groupby(['merchant_id'])['brand_label_ratio'].max().reset_index()

    data = data.merge(item_label_ratio_max, on='merchant_id', how='left')
    data = data.merge(cat_label_ratio_max, on='merchant_id', how='left')
    data = data.merge(brand_label_ratio_max, on='merchant_id', how='left')

    del train_ratio, merchant_label_ratio, item_label_ratio_max, cat_label_ratio_max, brand_label_ratio_max
    gc.collect()

    # 性别、年龄独热编码处理
    data = data.merge(user_info, on="user_id", how="left")

    temp = pd.get_dummies(data["age_range"], prefix="age", dtype='int32')
    temp2 = pd.get_dummies(data["gender"], prefix="gender", dtype='int32')

    data = pd.concat([data, temp, temp2], axis=1)
    data.drop(columns=["age_range", "gender"], inplace=True)

    del temp, temp2
    gc.collect()

    # 按user_id,merchant_id分组，购买天数>1则复购标记为1，反之为0
    groups_rb = user_log[user_log["action_type"] == 2].groupby(["user_id", "merchant_id"])
    temp_rb = groups_rb.time_stamp.nunique().reset_index().rename(columns={"time_stamp": "n_days"})
    temp_rb["label_um"] = [(1 if x > 1 else 0) for x in temp_rb["n_days"]]

    groups = user_log.groupby(["user_id"])
    # 日志数
    temp = groups.size().reset_index().rename(columns={0: "count_u"})
    data = pd.merge(data, temp, on="user_id", how="left")

    # 交互天数
    temp = groups.time_stamp.nunique().reset_index().rename(columns={"time_stamp": "days_u"})
    data = data.merge(temp, on="user_id", how="left")

    # 访问商品，品类，品牌，商家数
    temp = groups[['item_id', 'cat_id', 'merchant_id', 'brand_id']].nunique().reset_index().rename(columns={
        'item_id': 'item_count_u', 'cat_id': 'cat_count_u', 'merchant_id': 'merchant_count_u',
        'brand_id': 'brand_count_u'})
    data = data.merge(temp, on="user_id", how="left")

    # 各行为类型次数
    temp = groups['action_type'].value_counts().unstack().reset_index().rename(columns={
        0: 'view_count_u', 1: 'cart_count_u', 2: 'buy_count_u', 3: 'fav_count_u'})
    data = data.merge(temp, on="user_id", how="left")

    # 行为权重
    data['action_weight_u'] = (
                data['view_count_u'] * 0.1 + data['cart_count_u'] * 0.2 + data['fav_count_u'] * 0.2 + data[
            'buy_count_u'] * 0.5)

    # 统计购买点击比
    data["buy_view_ratio_u"] = data["buy_count_u"] / data["view_count_u"]
    # 统计购买收藏比
    data['buy_fav_ratio_u'] = data['buy_count_u'] / data['fav_count_u']
    # 统计购买加购比
    data['buy_cart_ratio_u'] = data['buy_count_u'] / data['cart_count_u']
    # 购买频率
    data['buy_freq_u'] = data['buy_count_u'] / data['count_u']

    # 复购率 = 复购过的商家数/购买过的总商家数
    temp = temp_rb.groupby(["user_id", "label_um"]).size().unstack(fill_value=0).reset_index()
    temp["repurchase_rate_u"] = temp[1] / (temp[0] + temp[1])
    data = data.merge(temp[["user_id", "repurchase_rate_u"]], on="user_id", how="left")

    # 购买量/购买商家数
    temp = user_log[user_log['action_type'] == 2].groupby(['user_id']).merchant_id.nunique().reset_index().rename(
        columns={'merchant_id': 'merchant_buy_count'})
    data = data.merge(temp, on='user_id', how='left')
    data['loyal_u'] = data['buy_count_u'] / data['merchant_buy_count']

    groups = user_log.groupby(["merchant_id"])

    # 日志数
    temp = groups.size().reset_index().rename(columns={0: "count_m"})
    data = data.merge(temp, on="merchant_id", how="left")

    # 交互天数
    temp = groups.time_stamp.nunique().reset_index().rename(columns={"time_stamp": "days_m"})
    data = data.merge(temp, on="merchant_id", how="left")

    # 访问商品，品类，品牌，用户数
    temp = groups[['item_id', 'cat_id', 'user_id', 'brand_id']].nunique().reset_index().rename(columns={
        'item_id': 'item_count_m', 'cat_id': 'cat_count_m', 'user_id': 'user_count_m', 'brand_id': 'brand_count_m'})
    data = data.merge(temp, on="merchant_id", how="left")

    # 各行为类型次数
    temp = groups.action_type.value_counts().unstack().reset_index().rename(columns={
        0: 'view_count_m', 1: 'cart_count_m', 2: 'buy_count_m', 3: 'fav_count_m'})
    data = data.merge(temp, on="merchant_id", how="left")

    # 行为权重
    data['action_weight_m'] = (
                data['view_count_m'] * 0.1 + data['cart_count_m'] * 0.2 + data['fav_count_m'] * 0.2 + data[
            'buy_count_m'] * 0.5)

    # 统计购买点击比
    data["buy_view_ratio_m"] = data["buy_count_m"] / data["view_count_m"]
    # 统计购买收藏比
    data['buy_fav_ratio_m'] = data['buy_count_m'] / data['fav_count_m']
    # 统计购买加购比
    data['buy_cart_ratio_m'] = data['buy_count_m'] / data['cart_count_m']
    # 购买频率
    data['buy_freq_m'] = data['buy_count_m'] / data['count_m']

    # 复购率
    temp = temp_rb.groupby(["merchant_id", "label_um"]).size().unstack(fill_value=0).reset_index()
    temp["repurchase_rate_m"] = temp[1] / (temp[0] + temp[1])
    data = data.merge(temp[["merchant_id", "repurchase_rate_m"]], on="merchant_id", how="left")

    # 购买量/购买用户数
    temp = user_log[user_log['action_type'] == 2].groupby(['merchant_id']).user_id.nunique().reset_index().rename(
        columns={'user_id': 'user_buy_count'})
    data = data.merge(temp, on='merchant_id', how='left')
    data['loyal_m'] = data['buy_count_m'] / data['user_buy_count']

    groups = user_log.groupby(['user_id', 'merchant_id'])

    # 日志数
    temp = groups.size().reset_index().rename(columns={0: "count_um"})
    data = data.merge(temp, on=["user_id", "merchant_id"], how="left")

    # 交互天数
    temp = groups.time_stamp.nunique().reset_index().rename(columns={"time_stamp": "days_um"})
    data = data.merge(temp, on=["user_id", "merchant_id"], how="left")

    # 访问商品，品类，品牌数
    temp = groups[['item_id', 'cat_id', 'brand_id']].nunique().reset_index().rename(columns={
        'item_id': 'item_count_um', 'cat_id': 'cat_count_um', 'brand_id': 'brand_count_um'})
    data = data.merge(temp, on=["user_id", "merchant_id"], how="left")

    # 各行为类型次数
    temp = groups.action_type.value_counts().unstack().reset_index().rename(columns={
        0: 'view_count_um', 1: 'cart_count_um', 2: 'buy_count_um', 3: 'fav_count_um'})
    data = data.merge(temp, on=["user_id", "merchant_id"], how="left")

    # 行为权重
    data['action_weight_um'] = (
                data['view_count_um'] * 0.1 + data['cart_count_um'] * 0.2 + data['fav_count_um'] * 0.2 + data[
            'buy_count_um'] * 0.5)
    # 统计购买点击比
    data["buy_view_ratio_um"] = data["buy_count_um"] / data["view_count_um"]
    # 统计购买收藏比
    data['buy_fav_ratio_um'] = data['buy_count_um'] / data['fav_count_um']
    # 统计购买加购比
    data['buy_cart_ratio_um'] = data['buy_count_um'] / data['cart_count_um']
    # 购买频率
    data['buy_freq_um'] = data['buy_count_um'] / data['count_um']

    # 交互点击比
    data['view_ratio'] = data['view_count_um'] / data['view_count_u']
    # 交互加购比
    data['cart_ratio'] = data['cart_count_um'] / data['cart_count_u']
    # 交互收藏比
    data['fav_ratio'] = data['fav_count_um'] / data['fav_count_u']
    # 交互购买比
    data['buy_ratio'] = data['buy_count_um'] / data['buy_count_u']
    # 交互比
    data['interaction_ratio'] = data['count_um'] / data['count_u']

    numerical_cols = data.select_dtypes(include=[np.number]).columns
    data[numerical_cols] = data[numerical_cols].fillna(0)

    gc.collect()
    data.to_csv("data/2/features1.csv", index=False)

In [ ]:
# process_data()

C:\Users\29421\AppData\Local\Temp\ipykernel_4360\3631602811.py:14: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  user_info["age_range"].fillna(0, inplace=True)
C:\Users\29421\AppData\Local\Temp\ipykernel_4360\3631602811.py:15: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For examp

In [ ]:
from sklearn.discriminant_analysis import StandardScaler
from sklearn.pipeline import make_pipeline
import torch
import pandas as pd
from pytabkit import TabM_D_Classifier, TabM_HPO_Classifier, LGBM_TD_Classifier
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from models.iwt_classifier import IWT_Classifier

data = pd.read_csv(f'data/2/features1.csv')
train = data[data["origin"] == "train"].drop(["origin"], axis=1)
test = data[data["origin"] == "test"].drop(["origin", "label"], axis=1)

X, Y = train.drop(['user_id', 'merchant_id', 'label'], axis=1), train['label']
X_test = test.drop(columns=['user_id', 'merchant_id'])

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
num_groups = X.shape[1]
num_features = X.shape[1]
avg_size = round(num_features // num_groups)
gidx_list = []
for k in range(num_groups):
    gidx_list.extend([k] * avg_size)
gidx_list.extend([num_groups - 1] * (num_features - avg_size * num_groups))
gidx = torch.tensor(gidx_list, dtype=torch.long, device=device)
sgidx = []
for kki in range(num_groups):
    idx = torch.where(gidx == kki)[0]
    sgidx.append(idx)

iwt_clf = IWT_Classifier(
    num_groups=len(sgidx),
    s=50,
    gidx=gidx,
    sgidx=sgidx,
    strategy='B',
    equalsize=True,
    verbose=False,
    draw_loss=True
)

pipeline = make_pipeline(
    StandardScaler(), LogisticRegression(C = 0.1, penalty = 'l1',solver='liblinear')#iwt_clf
)

estimators = [
    ('lgbm', LGBM_TD_Classifier(val_metric_name='1-auc_ovr_alt', random_state=42)),
    ('rf', RandomForestClassifier(n_estimators=10, random_state=42)),
]
clf = StackingClassifier(
    estimators=estimators, final_estimator=pipeline
)
clf = pipeline
clf.fit(X, Y)

test_pred = clf.predict_proba(X_test)[:, 1]

submission = test[['user_id', 'merchant_id']].copy()
submission['prob'] = test_pred
submission.to_csv('data/2/submission_tabM.csv', index=False)
print('Submission csv saved!')

图像已保存到: loss_history_lambda_0.8017953205361363.png
Final loss: 0.6893036961555481
Total iterations: 107
Submission csv saved!


In [ ]:
import numpy as np

d = {'alg_weights': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 14, 0, 0, 0, 0, 0, 16, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'sub_fit_params': [{'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm-mini', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.0005058942529815513), 'weight_decay': np.float64(0.0), 'n_blocks': np.int64(5), 'd_block': np.int64(480), 'dropout': np.float64(0.0), 'num_emb_type': 'pwl', 'd_embedding': np.int64(28), 'num_emb_n_bins': np.int64(96), 'sub_fit_params': {'sub_fit_params': [None]}}, {'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm-mini', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.0003370062728089647), 'weight_decay': np.float64(0.00025518969970339756), 'n_blocks': np.int64(5), 'd_block': np.int64(976), 'dropout': np.float64(0.1255057131396986), 'num_emb_type': 'pwl', 'd_embedding': np.int64(28), 'num_emb_n_bins': np.int64(23), 'sub_fit_params': {'sub_fit_params': [None]}}, {'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm-mini', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.0021399752156178284), 'weight_decay': np.float64(0.0), 'n_blocks': np.int64(2), 'd_block': np.int64(688), 'dropout': np.float64(0.0), 'num_emb_type': 'pwl', 'd_embedding': np.int64(12), 'num_emb_n_bins': np.int64(84), 'sub_fit_params': {'sub_fit_params': [None]}}, {'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm-mini', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.0011136343791654155), 'weight_decay': np.float64(0.0), 'n_blocks': np.int64(2), 'd_block': np.int64(160), 'dropout': np.float64(0.3109730730172545), 'num_emb_type': 'pwl', 'd_embedding': np.int64(20), 'num_emb_n_bins': np.int64(96), 'sub_fit_params': {'sub_fit_params': [None]}}, {'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm-mini', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.0003706377021434626), 'weight_decay': np.float64(0.0674066641717872), 'n_blocks': np.int64(5), 'd_block': np.int64(384), 'dropout': np.float64(0.37362818327260205), 'num_emb_type': 'pwl', 'd_embedding': np.int64(24), 'num_emb_n_bins': np.int64(79), 'sub_fit_params': {'sub_fit_params': [None]}}, {'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm-mini', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.00013313089945964177), 'weight_decay': np.float64(0.0), 'n_blocks': np.int64(4), 'd_block': np.int64(576), 'dropout': np.float64(0.4127963921140429), 'num_emb_type': 'pwl', 'd_embedding': np.int64(28), 'num_emb_n_bins': np.int64(83), 'sub_fit_params': {'sub_fit_params': [None]}}, {'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm-mini', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.000728482449630055), 'weight_decay': np.float64(0.0), 'n_blocks': np.int64(2), 'd_block': np.int64(800), 'dropout': np.float64(0.0), 'num_emb_type': 'pwl', 'd_embedding': np.int64(12), 'num_emb_n_bins': np.int64(55), 'sub_fit_params': {'sub_fit_params': [None]}}, {'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm-mini', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.0005420969545908613), 'weight_decay': np.float64(0.028783143186014805), 'n_blocks': np.int64(3), 'd_block': np.int64(896), 'dropout': np.float64(0.14411909219317454), 'num_emb_type': 'pwl', 'd_embedding': np.int64(20), 'num_emb_n_bins': np.int64(42), 'sub_fit_params': {'sub_fit_params': [None]}}, {'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm-mini', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.00019936661888626597), 'weight_decay': np.float64(0.0), 'n_blocks': np.int64(2), 'd_block': np.int64(864), 'dropout': np.float64(0.0), 'num_emb_type': 'pwl', 'd_embedding': np.int64(16), 'num_emb_n_bins': np.int64(27), 'sub_fit_params': {'sub_fit_params': [None]}}, {'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm-mini', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.0007561404492770056), 'weight_decay': np.float64(0.022028699054526424), 'n_blocks': np.int64(2), 'd_block': np.int64(384), 'dropout': np.float64(0.0), 'num_emb_type': 'pwl', 'd_embedding': np.int64(16), 'num_emb_n_bins': np.int64(112), 'sub_fit_params': {'sub_fit_params': [None]}}, {'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm-mini', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.0028038454523110893), 'weight_decay': np.float64(0.005695326800564199), 'n_blocks': np.int64(2), 'd_block': np.int64(336), 'dropout': np.float64(0.20949882117718738), 'num_emb_type': 'pwl', 'd_embedding': np.int64(20), 'num_emb_n_bins': np.int64(98), 'sub_fit_params': {'sub_fit_params': [None]}}, {'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm-mini', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.0029993695720154537), 'weight_decay': np.float64(0.023742083301699905), 'n_blocks': np.int64(4), 'd_block': np.int64(480), 'dropout': np.float64(0.0), 'num_emb_type': 'pwl', 'd_embedding': np.int64(32), 'num_emb_n_bins': np.int64(119), 'sub_fit_params': {'sub_fit_params': [None]}}, {'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm-mini', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.0008129109593390421), 'weight_decay': np.float64(0.0016376903232765726), 'n_blocks': np.int64(2), 'd_block': np.int64(560), 'dropout': np.float64(0.0), 'num_emb_type': 'pwl', 'd_embedding': np.int64(32), 'num_emb_n_bins': np.int64(116), 'sub_fit_params': {'sub_fit_params': [None]}}, {'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm-mini', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.0015910262500807941), 'weight_decay': np.float64(0.04959253028652543), 'n_blocks': np.int64(3), 'd_block': np.int64(944), 'dropout': np.float64(0.4822038937831297), 'num_emb_type': 'pwl', 'd_embedding': np.int64(8), 'num_emb_n_bins': np.int64(49), 'sub_fit_params': {'sub_fit_params': [None]}}, {'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm-mini', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.00019388822783584738), 'weight_decay': np.float64(0.0002884816325704595), 'n_blocks': np.int64(4), 'd_block': np.int64(208), 'dropout': np.float64(0.31585605378222414), 'num_emb_type': 'pwl', 'd_embedding': np.int64(20), 'num_emb_n_bins': np.int64(88), 'sub_fit_params': {'sub_fit_params': [None]}}, {'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm-mini', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.00024135314978969302), 'weight_decay': np.float64(0.0024471881561705243), 'n_blocks': np.int64(5), 'd_block': np.int64(944), 'dropout': np.float64(0.0744801481938846), 'num_emb_type': 'pwl', 'd_embedding': np.int64(28), 'num_emb_n_bins': np.int64(95), 'sub_fit_params': {'sub_fit_params': [None]}}, {'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm-mini', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.00015935816215787446), 'weight_decay': np.float64(0.0013737938502234532), 'n_blocks': np.int64(3), 'd_block': np.int64(656), 'dropout': np.float64(0.2540503336259255), 'num_emb_type': 'pwl', 'd_embedding': np.int64(12), 'num_emb_n_bins': np.int64(46), 'sub_fit_params': {'sub_fit_params': [None]}}, {'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm-mini', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.00063654518490572), 'weight_decay': np.float64(0.0), 'n_blocks': np.int64(4), 'd_block': np.int64(304), 'dropout': np.float64(0.1408249334797056), 'num_emb_type': 'pwl', 'd_embedding': np.int64(24), 'num_emb_n_bins': np.int64(115), 'sub_fit_params': {'sub_fit_params': [None]}}, {'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm-mini', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.00015599848043915948), 'weight_decay': np.float64(0.0005665939724584734), 'n_blocks': np.int64(3), 'd_block': np.int64(976), 'dropout': np.float64(0.0), 'num_emb_type': 'pwl', 'd_embedding': np.int64(32), 'num_emb_n_bins': np.int64(100), 'sub_fit_params': {'sub_fit_params': [None]}}, {'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm-mini', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.0024847500983596693), 'weight_decay': np.float64(0.05758459292273077), 'n_blocks': np.int64(5), 'd_block': np.int64(256), 'dropout': np.float64(0.0), 'num_emb_type': 'pwl', 'd_embedding': np.int64(28), 'num_emb_n_bins': np.int64(65), 'sub_fit_params': {'sub_fit_params': [None]}}, {'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm-mini', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.0023446004241301846), 'weight_decay': np.float64(0.00408963553628068), 'n_blocks': np.int64(5), 'd_block': np.int64(336), 'dropout': np.float64(0.18414188880295457), 'num_emb_type': 'pwl', 'd_embedding': np.int64(28), 'num_emb_n_bins': np.int64(28), 'sub_fit_params': {'sub_fit_params': [None]}}, {'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm-mini', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.00032723606486651755), 'weight_decay': np.float64(0.0), 'n_blocks': np.int64(3), 'd_block': np.int64(208), 'dropout': np.float64(0.0), 'num_emb_type': 'pwl', 'd_embedding': np.int64(20), 'num_emb_n_bins': np.int64(91), 'sub_fit_params': {'sub_fit_params': [None]}}, {'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm-mini', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.0014841671090472773), 'weight_decay': np.float64(0.0), 'n_blocks': np.int64(5), 'd_block': np.int64(960), 'dropout': np.float64(0.22519012395373705), 'num_emb_type': 'pwl', 'd_embedding': np.int64(28), 'num_emb_n_bins': np.int64(109), 'sub_fit_params': {'sub_fit_params': [None]}}, {'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm-mini', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.0001228978992492114), 'weight_decay': np.float64(0.0002497312225508937), 'n_blocks': np.int64(4), 'd_block': np.int64(848), 'dropout': np.float64(0.0), 'num_emb_type': 'pwl', 'd_embedding': np.int64(16), 'num_emb_n_bins': np.int64(15), 'sub_fit_params': {'sub_fit_params': [None]}}, {'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm-mini', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.0007105013141226399), 'weight_decay': np.float64(0.0), 'n_blocks': np.int64(3), 'd_block': np.int64(784), 'dropout': np.float64(0.2909560584035028), 'num_emb_type': 'pwl', 'd_embedding': np.int64(24), 'num_emb_n_bins': np.int64(62), 'sub_fit_params': {'sub_fit_params': [None]}}, {'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm-mini', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.0011450282628469696), 'weight_decay': np.float64(0.0006476452727183112), 'n_blocks': np.int64(5), 'd_block': np.int64(608), 'dropout': np.float64(0.0), 'num_emb_type': 'pwl', 'd_embedding': np.int64(28), 'num_emb_n_bins': np.int64(5), 'sub_fit_params': {'sub_fit_params': [None]}}, {'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm-mini', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.00022986824115017986), 'weight_decay': np.float64(0.0), 'n_blocks': np.int64(4), 'd_block': np.int64(672), 'dropout': np.float64(0.0), 'num_emb_type': 'pwl', 'd_embedding': np.int64(32), 'num_emb_n_bins': np.int64(2), 'sub_fit_params': {'sub_fit_params': [None]}}, {'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm-mini', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.0005717152800280225), 'weight_decay': np.float64(0.028322308962600466), 'n_blocks': np.int64(2), 'd_block': np.int64(1024), 'dropout': np.float64(0.0), 'num_emb_type': 'pwl', 'd_embedding': np.int64(32), 'num_emb_n_bins': np.int64(47), 'sub_fit_params': {'sub_fit_params': [None]}}, {'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm-mini', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.00010394798647461192), 'weight_decay': np.float64(0.0047461973363614705), 'n_blocks': np.int64(3), 'd_block': np.int64(192), 'dropout': np.float64(0.4806247413279848), 'num_emb_type': 'pwl', 'd_embedding': np.int64(8), 'num_emb_n_bins': np.int64(33), 'sub_fit_params': {'sub_fit_params': [None]}}, {'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm-mini', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.0027282180354502887), 'weight_decay': np.float64(0.0), 'n_blocks': np.int64(3), 'd_block': np.int64(624), 'dropout': np.float64(0.0), 'num_emb_type': 'pwl', 'd_embedding': np.int64(12), 'num_emb_n_bins': np.int64(69), 'sub_fit_params': {'sub_fit_params': [None]}}, {'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm-mini', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.0003340974836459999), 'weight_decay': np.float64(0.0), 'n_blocks': np.int64(5), 'd_block': np.int64(432), 'dropout': np.float64(0.0), 'num_emb_type': 'pwl', 'd_embedding': np.int64(20), 'num_emb_n_bins': np.int64(38), 'sub_fit_params': {'sub_fit_params': [None]}}, {'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm-mini', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.0005529145392225712), 'weight_decay': np.float64(0.002293695957254325), 'n_blocks': np.int64(5), 'd_block': np.int64(544), 'dropout': np.float64(0.23096641240310473), 'num_emb_type': 'pwl', 'd_embedding': np.int64(32), 'num_emb_n_bins': np.int64(50), 'sub_fit_params': {'sub_fit_params': [None]}}, {'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm-mini', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.00016457863882165822), 'weight_decay': np.float64(0.00020768129523103025), 'n_blocks': np.int64(2), 'd_block': np.int64(928), 'dropout': np.float64(0.0), 'num_emb_type': 'pwl', 'd_embedding': np.int64(12), 'num_emb_n_bins': np.int64(42), 'sub_fit_params': {'sub_fit_params': [None]}}, {'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm-mini', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.0015025617074357945), 'weight_decay': np.float64(0.0), 'n_blocks': np.int64(3), 'd_block': np.int64(240), 'dropout': np.float64(0.0), 'num_emb_type': 'pwl', 'd_embedding': np.int64(20), 'num_emb_n_bins': np.int64(33), 'sub_fit_params': {'sub_fit_params': [None]}}, {'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm-mini', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.002321555640112252), 'weight_decay': np.float64(0.0013723424702556703), 'n_blocks': np.int64(3), 'd_block': np.int64(624), 'dropout': np.float64(0.0), 'num_emb_type': 'pwl', 'd_embedding': np.int64(32), 'num_emb_n_bins': np.int64(79), 'sub_fit_params': {'sub_fit_params': [None]}}, {'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm-mini', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.0005403969955943344), 'weight_decay': np.float64(0.08976518796993564), 'n_blocks': np.int64(4), 'd_block': np.int64(320), 'dropout': np.float64(0.0), 'num_emb_type': 'pwl', 'd_embedding': np.int64(12), 'num_emb_n_bins': np.int64(7), 'sub_fit_params': {'sub_fit_params': [None]}}, {'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm-mini', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.00019686923836294835), 'weight_decay': np.float64(0.0), 'n_blocks': np.int64(2), 'd_block': np.int64(768), 'dropout': np.float64(0.0), 'num_emb_type': 'pwl', 'd_embedding': np.int64(8), 'num_emb_n_bins': np.int64(25), 'sub_fit_params': {'sub_fit_params': [None]}}, {'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm-mini', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.0002752675962600818), 'weight_decay': np.float64(0.0), 'n_blocks': np.int64(2), 'd_block': np.int64(256), 'dropout': np.float64(0.17876431194975523), 'num_emb_type': 'pwl', 'd_embedding': np.int64(8), 'num_emb_n_bins': np.int64(6), 'sub_fit_params': {'sub_fit_params': [None]}}, {'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm-mini', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.0006547230807469514), 'weight_decay': np.float64(0.0), 'n_blocks': np.int64(3), 'd_block': np.int64(320), 'dropout': np.float64(0.08797390037169411), 'num_emb_type': 'pwl', 'd_embedding': np.int64(32), 'num_emb_n_bins': np.int64(123), 'sub_fit_params': {'sub_fit_params': [None]}}, {'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm-mini', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.0005320416197654371), 'weight_decay': np.float64(0.0007260189567411681), 'n_blocks': np.int64(3), 'd_block': np.int64(336), 'dropout': np.float64(0.0), 'num_emb_type': 'pwl', 'd_embedding': np.int64(32), 'num_emb_n_bins': np.int64(71), 'sub_fit_params': {'sub_fit_params': [None]}}, {'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm-mini', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.0003067647210595889), 'weight_decay': np.float64(0.0), 'n_blocks': np.int64(2), 'd_block': np.int64(992), 'dropout': np.float64(0.08437339796340931), 'num_emb_type': 'pwl', 'd_embedding': np.int64(8), 'num_emb_n_bins': np.int64(106), 'sub_fit_params': {'sub_fit_params': [None]}}, {'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm-mini', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.00025489141069185334), 'weight_decay': np.float64(0.0), 'n_blocks': np.int64(5), 'd_block': np.int64(592), 'dropout': np.float64(0.0), 'num_emb_type': 'pwl', 'd_embedding': np.int64(12), 'num_emb_n_bins': np.int64(115), 'sub_fit_params': {'sub_fit_params': [None]}}, {'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm-mini', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.0023391419274724577), 'weight_decay': np.float64(0.00011318938127325326), 'n_blocks': np.int64(4), 'd_block': np.int64(752), 'dropout': np.float64(0.3583983454742526), 'num_emb_type': 'pwl', 'd_embedding': np.int64(16), 'num_emb_n_bins': np.int64(116), 'sub_fit_params': {'sub_fit_params': [None]}}, {'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm-mini', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.00029601315902864217), 'weight_decay': np.float64(0.0), 'n_blocks': np.int64(2), 'd_block': np.int64(176), 'dropout': np.float64(0.43157192107173936), 'num_emb_type': 'pwl', 'd_embedding': np.int64(32), 'num_emb_n_bins': np.int64(69), 'sub_fit_params': {'sub_fit_params': [None]}}, {'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm-mini', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.000516699118415651), 'weight_decay': np.float64(0.0), 'n_blocks': np.int64(5), 'd_block': np.int64(352), 'dropout': np.float64(0.3103646687676049), 'num_emb_type': 'pwl', 'd_embedding': np.int64(20), 'num_emb_n_bins': np.int64(84), 'sub_fit_params': {'sub_fit_params': [None]}}, {'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm-mini', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.0008254687238451864), 'weight_decay': np.float64(0.0), 'n_blocks': np.int64(2), 'd_block': np.int64(512), 'dropout': np.float64(0.0), 'num_emb_type': 'pwl', 'd_embedding': np.int64(8), 'num_emb_n_bins': np.int64(82), 'sub_fit_params': {'sub_fit_params': [None]}}, {'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm-mini', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.0001496343969471598), 'weight_decay': np.float64(0.00018462680481859064), 'n_blocks': np.int64(4), 'd_block': np.int64(768), 'dropout': np.float64(0.4612777988014327), 'num_emb_type': 'pwl', 'd_embedding': np.int64(16), 'num_emb_n_bins': np.int64(126), 'sub_fit_params': {'sub_fit_params': [None]}}, {'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm-mini', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.00010696634102056284), 'weight_decay': np.float64(0.0), 'n_blocks': np.int64(3), 'd_block': np.int64(256), 'dropout': np.float64(0.0), 'num_emb_type': 'pwl', 'd_embedding': np.int64(28), 'num_emb_n_bins': np.int64(59), 'sub_fit_params': {'sub_fit_params': [None]}}, {'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm-mini', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.00038967826291135834), 'weight_decay': np.float64(0.000629793603282554), 'n_blocks': np.int64(2), 'd_block': np.int64(560), 'dropout': np.float64(0.0), 'num_emb_type': 'pwl', 'd_embedding': np.int64(16), 'num_emb_n_bins': np.int64(78), 'sub_fit_params': {'sub_fit_params': [None]}}, {'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm-mini', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.0007873707358543825), 'weight_decay': np.float64(0.013077728680078912), 'n_blocks': np.int64(3), 'd_block': np.int64(704), 'dropout': np.float64(0.0), 'num_emb_type': 'pwl', 'd_embedding': np.int64(24), 'num_emb_n_bins': np.int64(101), 'sub_fit_params': {'sub_fit_params': [None]}}]}

w_l = d['alg_weights']
for i, w in enumerate(w_l):
    if w != 0:
        print(f'Weight {i}: {w}')

print(d['sub_fit_params'][26])

w26_14 = {'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm-mini', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.00022986824115017986), 'weight_decay': np.float64(0.0), 'n_blocks': np.int64(4), 'd_block': np.int64(672), 'dropout': np.float64(0.0), 'num_emb_type': 'pwl', 'd_embedding': np.int64(32), 'num_emb_n_bins': np.int64(2), 'sub_fit_params': {'sub_fit_params': [None]}}

w32_16 = {'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm-mini', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.00016457863882165822), 'weight_decay': np.float64(0.00020768129523103025), 'n_blocks': np.int64(2), 'd_block': np.int64(928), 'dropout': np.float64(0.0), 'num_emb_type': 'pwl', 'd_embedding': np.int64(12), 'num_emb_n_bins': np.int64(42), 'sub_fit_params': {'sub_fit_params': [None]}}

: 

tab-mini weight[16  0  8  3  4  0  1  2  0  0] 0.6959399114348 5k 10steps

*self.fit_params[0]={'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm-mini', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.0001334456483981059), 'weight_decay': np.float64(0.012340787659576354), 'n_blocks': np.int64(5), 'd_block': np.int64(912), 'dropout': np.float64(0.0), 'num_emb_type': 'pwl', 'd_embedding': np.int64(32), 'num_emb_n_bins': np.int64(2)} 10k 0.6956

self.fit_params[0]={'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm-mini', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.00021816266197247693), 'weight_decay': np.float64(0.0), 'n_blocks': np.int64(3), 'd_block': np.int64(640), 'dropout': np.float64(0.33055586392588004), 'num_emb_type': 'pwl', 'd_embedding': np.int64(32), 'num_emb_n_bins': np.int64(26)}

*self.fit_params[0]={'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm-mini', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.00022029015158645592), 'weight_decay': np.float64(0.0030179664067750372), 'n_blocks': np.int64(3), 'd_block': np.int64(432), 'dropout': np.float64(0.3544382638112154), 'num_emb_type': 'pwl', 'd_embedding': np.int64(28), 'num_emb_n_bins': np.int64(127)}

*self.fit_params[0]={'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm-mini', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.00013795505877650676), 'weight_decay': np.float64(0.0), 'n_blocks': np.int64(3), 'd_block': np.int64(864), 'dropout': np.float64(0.08366780807852209), 'num_emb_type': 'pwl', 'd_embedding': np.int64(12), 'num_emb_n_bins': np.int64(5)}

*self.fit_params[0]={'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm-mini', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.001048334934585524), 'weight_decay': np.float64(0.0), 'n_blocks': np.int64(5), 'd_block': np.int64(432), 'dropout': np.float64(0.0), 'num_emb_type': 'pwl', 'd_embedding': np.int64(12), 'num_emb_n_bins': np.int64(114)}

self.fit_params[0]={'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm-mini', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.0006668459082109882), 'weight_decay': np.float64(0.0), 'n_blocks': np.int64(3), 'd_block': np.int64(192), 'dropout': np.float64(0.0), 'num_emb_type': 'pwl', 'd_embedding': np.int64(28), 'num_emb_n_bins': np.int64(121)}

*self.fit_params[0]={'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm-mini', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.0003276095002985988), 'weight_decay': np.float64(0.00431990661862382), 'n_blocks': np.int64(2), 'd_block': np.int64(928), 'dropout': np.float64(0.0), 'num_emb_type': 'pwl', 'd_embedding': np.int64(32), 'num_emb_n_bins': np.int64(57)}

*self.fit_params[0]={'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm-mini', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.001492706280794318), 'weight_decay': np.float64(0.0), 'n_blocks': np.int64(4), 'd_block': np.int64(816), 'dropout': np.float64(0.4826120323148692), 'num_emb_type': 'pwl', 'd_embedding': np.int64(8), 'num_emb_n_bins': np.int64(82)}

self.fit_params[0]={'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm-mini', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.002515274885156232), 'weight_decay': np.float64(0.0), 'n_blocks': np.int64(2), 'd_block': np.int64(864), 'dropout': np.float64(0.10326996792916132), 'num_emb_type': 'pwl', 'd_embedding': np.int64(8), 'num_emb_n_bins': np.int64(52)}

self.fit_params[0]={'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm-mini', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.0014449145634601243), 'weight_decay': np.float64(0.0009636806325505294), 'n_blocks': np.int64(4), 'd_block': np.int64(448), 'dropout': np.float64(0.0), 'num_emb_type': 'pwl', 'd_embedding': np.int64(16), 'num_emb_n_bins': np.int64(78)}

=======================================================================================================================================

tabm weight[1 2 0 3 0 0 8 8 0 0] 0.6941 10steps

*self.fit_params[0]={'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.00013754583614053662), 'weight_decay': np.float64(0.0015053977006493477), 'n_blocks': np.int64(2), 'd_block': np.int64(512), 'dropout': np.float64(0.21324884955756607), 'num_emb_type': 'pwl', 'd_embedding': np.int64(8), 'num_emb_n_bins': np.int64(58)}

*self.fit_params[0]={'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.0003419510923533141), 'weight_decay': np.float64(0.0), 'n_blocks': np.int64(5), 'd_block': np.int64(928), 'dropout': np.float64(0.0), 'num_emb_type': 'pwl', 'd_embedding': np.int64(32), 'num_emb_n_bins': np.int64(10)}

self.fit_params[0]={'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.0015543461230552596), 'weight_decay': np.float64(0.0), 'n_blocks': np.int64(3), 'd_block': np.int64(272), 'dropout': np.float64(0.291544993424212), 'num_emb_type': 'pwl', 'd_embedding': np.int64(8), 'num_emb_n_bins': np.int64(34)}

*self.fit_params[0]={'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.0018594110485356079), 'weight_decay': np.float64(0.0), 'n_blocks': np.int64(5), 'd_block': np.int64(432), 'dropout': np.float64(0.09514423012628914), 'num_emb_type': 'pwl', 'd_embedding': np.int64(28), 'num_emb_n_bins': np.int64(125)}

self.fit_params[0]={'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.0001249171880339904), 'weight_decay': np.float64(0.0), 'n_blocks': np.int64(4), 'd_block': np.int64(704), 'dropout': np.float64(0.2808460055312869), 'num_emb_type': 'pwl', 'd_embedding': np.int64(32), 'num_emb_n_bins': np.int64(32)}

self.fit_params[0]={'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.0024293963603361076), 'weight_decay': np.float64(0.013370890566153053), 'n_blocks': np.int64(3), 'd_block': np.int64(928), 'dropout': np.float64(0.0), 'num_emb_type': 'pwl', 'd_embedding': np.int64(20), 'num_emb_n_bins': np.int64(97)}

*self.fit_params[0]={'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.001427730613920002), 'weight_decay': np.float64(0.00039996334669144577), 'n_blocks': np.int64(3), 'd_block': np.int64(1024), 'dropout': np.float64(0.1759765320133561), 'num_emb_type': 'pwl', 'd_embedding': np.int64(8), 'num_emb_n_bins': np.int64(23)}

*self.fit_params[0]={'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.00017803074484857816), 'weight_decay': np.float64(0.0), 'n_blocks': np.int64(4), 'd_block': np.int64(896), 'dropout': np.float64(0.07276177983723575), 'num_emb_type': 'pwl', 'd_embedding': np.int64(28), 'num_emb_n_bins': np.int64(14)}

self.fit_params[0]={'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.0008480921904553091), 'weight_decay': np.float64(0.0), 'n_blocks': np.int64(3), 'd_block': np.int64(176), 'dropout': np.float64(0.14206798398187792), 'num_emb_type': 'pwl', 'd_embedding': np.int64(32), 'num_emb_n_bins': np.int64(20)} 10k 0.6950

self.fit_params[0]={'batch_size': 'auto', 'patience': 16, 'allow_amp': False, 'arch_type': 'tabm', 'tabm_k': 32, 'share_training_batches': False, 'lr': np.float64(0.002099242883105876), 'weight_decay': np.float64(0.002385878732091997), 'n_blocks': np.int64(3), 'd_block': np.int64(848), 'dropout': np.float64(0.0), 'num_emb_type': 'pwl', 'd_embedding': np.int64(20), 'num_emb_n_bins': np.int64(109)}